##INSTALL + IMPORTS

In [1]:
# 1) INSTALL + IMPORTS
# ============================================================
!pip install pymupdf sentence-transformers scikit-learn spacy rarfile -q
!python -m spacy download en_core_web_sm -q
!apt-get install -y unrar -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 67.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 109.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Reading package lists...
Building dependency tree...
Reading state information...
unrar is already the newest version (1:6.1.5-1ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.


In [2]:
import os, re, zipfile, warnings, rarfile, pymupdf, pandas as pd, numpy as np
from google.colab import drive, files
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
warnings.filterwarnings("ignore")

##MOUNT DRIVE + DATA SPLIT

In [3]:
# ============================================================
# 2) MOUNT DRIVE + DATA SPLIT
# ============================================================
drive.mount('/content/drive')

rar_path = "/content/drive/MyDrive/job_descriptions.rar"
extract_path = "/content/dataset"
os.makedirs(extract_path, exist_ok=True)

with rarfile.RarFile(rar_path) as rf:
    rf.extractall(extract_path)

jobs_full = pd.read_csv("/content/dataset/job_descriptions.csv")

company_cols = [
    "Job Id","Company","Company Profile","Company Size",
    "location","Country","latitude","longitude",
    "Contact Person","Contact"
]

companies = jobs_full[company_cols]
companies.to_csv("/content/Companies.csv", index=False)

job_cols = [c for c in jobs_full.columns if c not in company_cols] + ["Job Id"]
jobs = jobs_full[job_cols].copy()
jobs["Date"] = pd.to_datetime(jobs["Job Posting Date"], format="%Y-%m-%d", errors="coerce")
jobs.drop(columns=["Job Posting Date"], inplace=True)
jobs.to_csv("/content/Jobs.csv", index=False)

with zipfile.ZipFile("/content/drive/MyDrive/Jobs_Companies.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write("/content/Jobs.csv", "Jobs.csv")
    zf.write("/content/Companies.csv", "Companies.csv")



Mounted at /content/drive


##LOAD MODEL + HELPERS

In [4]:
# ============================================================1
# 3) LOAD MODEL + HELPERS
# ============================================================
model = SentenceTransformer("all-MiniLM-L6-v2")

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\n]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def read_pdf(pdf_path):
    doc = pymupdf.open(pdf_path)
    return " ".join(page.get_text() for page in doc).strip()

def read_cvs(folder):
    names = os.listdir(folder)
    texts = [read_pdf(os.path.join(folder, name)) for name in names]
    return texts, names

def extract_skills(skill_str):
    clean = re.sub(r'\([^)]*\)', '', str(skill_str))
    parts = re.split(r'(?<=[a-z])\s+(?=[A-Z])', clean)
    result = []
    for s in parts:
        s = re.sub(r'[^a-z0-9\s/+#.-]', '', s.strip().lower()).strip()
        if 3 < len(s) <= 60:
            result.append(s)
    return result

def rank(names, scores):
    return sorted([(n, round(float(s)*100, 2)) for n, s in zip(names, scores)],
                  key=lambda x: x[1], reverse=True)

def baseline_scores(query, docs):
    vectorizer = TfidfVectorizer()
    X = vectorizer.fit_transform([preprocess(d) for d in docs] + [preprocess(query)])
    return cosine_similarity(X[-1], X[:-1])[0]

def semantic_scores(query, docs):
    q = model.encode([query])
    d = model.encode(docs)
    return cosine_similarity(q, d)[0]



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

... (3 outputs left out)

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

##EMPLOYER MODE

In [5]:

# ============================================================2
# 4) EMPLOYER MODE
# ============================================================
def employer_semantic(job_description, folder):
    cv_texts, cv_names = read_cvs(folder)
    return rank(cv_names, semantic_scores(job_description, cv_texts))

def employer_baseline(job_description, folder):
    cv_texts, cv_names = read_cvs(folder)
    return rank(cv_names, baseline_scores(job_description, cv_texts))

def compare_employer(job_description, folder, top_n=10):
    sem = pd.DataFrame(employer_semantic(job_description, folder)[:top_n], columns=["CV", "Semantic Score"])
    base = pd.DataFrame(employer_baseline(job_description, folder)[:top_n], columns=["CV", "Baseline Score"])
    return pd.merge(sem, base, on="CV", how="outer")


def employer_error_analysis(results, expected_relevant, top_n=10):
    rows = []
    for i, (cv, score) in enumerate(results[:top_n], 1):
        rows.append({
            "Rank": i,
            "CV": cv,
            "Score": score,
            "Relevant?": cv in expected_relevant,
            "Error Type": "Correct" if cv in expected_relevant else "False Positive"
        })
    return pd.DataFrame(rows)



##JOB SEEKER MODE

In [6]:
# ============================================================3
# 5) JOB SEEKER MODE
# ============================================================
print("📤 Upload your CV")
uploaded = files.upload()
cv_name = list(uploaded.keys())[0]
cv_path = f"/content/{cv_name}"
raw_cv_text = read_pdf(cv_path)

all_titles = jobs["Job Title"].dropna().unique().tolist()
cv_emb = model.encode(raw_cv_text, convert_to_tensor=True)
title_emb = model.encode(all_titles, convert_to_tensor=True)
title_scores = util.cos_sim(cv_emb, title_emb)[0]

top3_idx = title_scores.topk(3).indices.tolist()
top3_titles = [all_titles[i] for i in top3_idx]
top3_scores = [round(float(title_scores[i]) * 100, 2) for i in top3_idx]

print("Top 3 Titles:")
for t, s in zip(top3_titles, top3_scores):
    print(f"- {t}: {s}%")

best_title = top3_titles[0]

filtered_jobs = jobs[
    (jobs["Job Title"] == best_title) &
    ((jobs["Preference"] == "Male") | (jobs["Preference"] == "Female") | (jobs["Preference"] == "Both"))
]

def recommend_semantic(cv_text, filtered_jobs):
    docs = filtered_jobs["Job Description"].fillna("").tolist()
    scores = semantic_scores(cv_text, docs)
    result = filtered_jobs[["Job Title","Job Description","Salary Range","Work Type","Experience","skills","Benefits","Responsibilities"]].copy()
    result["Match_Score"] = [round(float(s)*100, 2) for s in scores]
    return result.sort_values("Match_Score", ascending=False).reset_index(drop=True)

def recommend_baseline(cv_text, filtered_jobs):
    docs = filtered_jobs["Job Description"].fillna("").tolist()
    scores = baseline_scores(cv_text, docs)
    result = filtered_jobs[["Job Title","Job Description","Salary Range","Work Type","Experience","skills","Benefits","Responsibilities"]].copy()
    result["Match_Score"] = [round(float(s)*100, 2) for s in scores]
    return result.sort_values("Match_Score", ascending=False).reset_index(drop=True)

semantic_jobs = recommend_semantic(raw_cv_text, filtered_jobs)
baseline_jobs = recommend_baseline(raw_cv_text, filtered_jobs)

print("🏆 Semantic Top 10")
display(semantic_jobs[["Job Title","Salary Range","Work Type","Experience","Match_Score"]].head(10))

print("🏆 Baseline Top 10")
display(baseline_jobs[["Job Title","Salary Range","Work Type","Experience","Match_Score"]].head(10))

def compare_jobseeker(semantic_df, baseline_df, top_n=10):
    sem = semantic_df[["Job Title","Match_Score"]].head(top_n).rename(columns={"Match_Score":"Semantic Score"})
    base = baseline_df[["Job Title","Match_Score"]].head(top_n).rename(columns={"Match_Score":"Baseline Score"})
    return pd.merge(sem, base, on="Job Title", how="outer")

comparison_jobs = compare_jobseeker(semantic_jobs, baseline_jobs)
display(comparison_jobs)


📤 Upload your CV


Saving Menna Ahmed Amer(CV3).pdf to Menna Ahmed Amer(CV3).pdf
Top 3 Titles:
- Data Scientist: 56.31%
- Data Analyst: 54.82%
- Data Engineer: 52.92%
🏆 Semantic Top 10


,Job Title,Salary Range,Work Type,Experience,Match_Score
0,Data Scientist,$65K-$119K,Intern,1 to 9 Years,52.43
1,Data Scientist,$55K-$111K,Temporary,4 to 9 Years,52.43
2,Data Scientist,$63K-$102K,Intern,1 to 8 Years,52.43
3,Data Scientist,$61K-$98K,Contract,0 to 9 Years,52.43
4,Data Scientist,$65K-$107K,Intern,4 to 14 Years,52.43
5,Data Scientist,$57K-$114K,Temporary,2 to 8 Years,52.43
6,Data Scientist,$65K-$105K,Contract,2 to 10 Years,52.43
7,Data Scientist,$56K-$98K,Intern,0 to 12 Years,52.43
8,Data Scientist,$55K-$126K,Intern,0 to 12 Years,52.43
9,Data Scientist,$62K-$105K,Temporary,2 to 10 Years,52.43


🏆 Baseline Top 10


,Job Title,Salary Range,Work Type,Experience,Match_Score
0,Data Scientist,$65K-$119K,Intern,1 to 9 Years,12.19
1,Data Scientist,$55K-$111K,Temporary,4 to 9 Years,12.19
2,Data Scientist,$63K-$102K,Intern,1 to 8 Years,12.19
3,Data Scientist,$61K-$98K,Contract,0 to 9 Years,12.19
4,Data Scientist,$65K-$107K,Intern,4 to 14 Years,12.19
5,Data Scientist,$57K-$114K,Temporary,2 to 8 Years,12.19
6,Data Scientist,$65K-$105K,Contract,2 to 10 Years,12.19
7,Data Scientist,$56K-$98K,Intern,0 to 12 Years,12.19
8,Data Scientist,$55K-$126K,Intern,0 to 12 Years,12.19
9,Data Scientist,$62K-$105K,Temporary,2 to 10 Years,12.19


,Job Title,Semantic Score,Baseline Score
0,Data Scientist,52.43,12.19
1,Data Scientist,52.43,12.19
2,Data Scientist,52.43,12.19
3,Data Scientist,52.43,12.19
4,Data Scientist,52.43,12.19
...,...,...,...
95,Data Scientist,52.43,12.19
96,Data Scientist,52.43,12.19
97,Data Scientist,52.43,12.19
98,Data Scientist,52.43,12.19


##SKILL GAP ANALYSIS

In [7]:
# ============================================================4
# 6) SKILL GAP ANALYSIS
# ============================================================
all_job_skills = set()
for s in filtered_jobs["skills"].dropna().tolist():
    for skill in extract_skills(s):
        all_job_skills.add(skill)

cv_skills = sorted([skill for skill in all_job_skills if skill in raw_cv_text.lower()])
job_required_skills = all_job_skills
cv_skills_set = set(cv_skills)
matched_skills = cv_skills_set & job_required_skills
missing_skills = job_required_skills - cv_skills_set
match_pct = round(len(matched_skills) / len(job_required_skills) * 100, 1) if job_required_skills else 0

print("✅ Skills you HAVE:")
for s in sorted(matched_skills):
    print("✅", s)

print("\n❌ Skills you are MISSING:")
for s in sorted(missing_skills)[:25]:
    print("❌", s)

print(f"\n🎯 Skill Match Rate: {match_pct}%")

✅ Skills you HAVE:
✅ data preprocessing
✅ deep learning
✅ model evaluation

❌ Skills you are MISSING:
❌ data integration
❌ data warehousing
❌ database management
❌ etl  big data technologies
❌ machine learning algorithms
❌ python programming

🎯 Skill Match Rate: 33.3%


##MODEL COMPARISON

In [8]:
# ============================================================4
# 1) MODEL COMPARISON
# ============================================================
def compare_jobseeker_models(semantic_df, baseline_df, top_n=10):
    sem = semantic_df[["Job Title", "Match_Score"]].head(top_n).copy()
    sem["Semantic Rank"] = range(1, len(sem) + 1)
    sem.rename(columns={"Match_Score": "Semantic Score"}, inplace=True)

    base = baseline_df[["Job Title", "Match_Score"]].head(top_n).copy()
    base["Baseline Rank"] = range(1, len(base) + 1)
    base.rename(columns={"Match_Score": "Baseline Score"}, inplace=True)

    comparison_df = pd.merge(sem, base, on="Job Title", how="outer")
    return comparison_df



In [9]:

# ============================================================4
# 5) RUN MODEL COMPARISON
# ============================================================
comparison_df = compare_jobseeker_models(semantic_jobs, baseline_jobs, top_n=10)

print("📊 Model Comparison")
display(comparison_df)

📊 Model Comparison


,Job Title,Semantic Score,Semantic Rank,Baseline Score,Baseline Rank
0,Data Scientist,52.43,1,12.19,1
1,Data Scientist,52.43,1,12.19,2
2,Data Scientist,52.43,1,12.19,3
3,Data Scientist,52.43,1,12.19,4
4,Data Scientist,52.43,1,12.19,5
...,...,...,...,...,...
95,Data Scientist,52.43,10,12.19,6
96,Data Scientist,52.43,10,12.19,7
97,Data Scientist,52.43,10,12.19,8
98,Data Scientist,52.43,10,12.19,9


##SHOW JOB TEXT FOR MANUAL LABELING

In [10]:
# ============================================================5
# 2) SHOW JOB TEXT FOR MANUAL LABELING
# ============================================================
def show_jobs_for_labeling(results_df, top_n=5, text_chars=500):
    top_results = results_df.head(top_n).reset_index(drop=True)

    for i, row in top_results.iterrows():
        print("=" * 80)
        print(f"Rank: {i+1}")
        print(f"Job Title: {row['Job Title']}")
        print(f"Score: {row['Match_Score']}")
        print("-" * 80)
        print("Job Description:")
        print(str(row["Job Description"])[:text_chars])
        print()

In [11]:
# ============================================================5
# 6) SHOW TEXT TO LABEL MANUALLY
# ============================================================
print("🔹 Semantic Results")
show_jobs_for_labeling(semantic_jobs, top_n=5, text_chars=500)

print("🔹 Baseline Results")
show_jobs_for_labeling(baseline_jobs, top_n=5, text_chars=500)

🔹 Semantic Results
Rank: 1
Job Title: Data Scientist
Score: 52.43
--------------------------------------------------------------------------------
Job Description:
Machine Learning Engineers develop machine learning models and algorithms, working on tasks like data preprocessing, model training, and deployment.

Rank: 2
Job Title: Data Scientist
Score: 52.43
--------------------------------------------------------------------------------
Job Description:
Machine Learning Engineers develop machine learning models and algorithms, working on tasks like data preprocessing, model training, and deployment.

Rank: 3
Job Title: Data Scientist
Score: 52.43
--------------------------------------------------------------------------------
Job Description:
Machine Learning Engineers develop machine learning models and algorithms, working on tasks like data preprocessing, model training, and deployment.

Rank: 4
Job Title: Data Scientist
Score: 52.43
-------------------------------------------------

## ERROR ANALYSIS

In [12]:
# ============================================================6
# 3) ERROR ANALYSIS
# relevance_labels:
# 1 = relevant
# 0 = not relevant
# ============================================================
def error_analysis(results_df, relevance_labels, model_name, top_n=10):
    rows = []
    top_results = results_df.head(top_n).reset_index(drop=True)

    if len(relevance_labels) != len(top_results):
        raise ValueError("Length of relevance_labels must equal top_n")

    for i, row in top_results.iterrows():
        is_relevant = relevance_labels[i]

        if is_relevant == 1:
            error_type = "Correct"
            reason = "Relevant to CV based on job description"
        else:
            error_type = "False Positive"
            reason = "Not relevant to CV based on job description"

        rows.append({
            "Model": model_name,
            "Rank": i + 1,
            "Job Title": row["Job Title"],
            "Score": row["Match_Score"],
            "Relevant?": is_relevant,
            "Error Type": error_type,
            "Reason": reason
        })

    return pd.DataFrame(rows)



##METRICS

In [13]:

# ============================================================6
# 4) METRICS
# y_true = relevance_labels
# y_pred = [1,1,1,...] لأن كل النتائج المعروضة اتوصى بها
# ============================================================
def classification_metrics(y_true, y_pred):
    tp = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 1)
    fp = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 1)
    fn = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 0)

    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

    return {
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1 Score": round(f1, 4)
    }



##Final Error Analysis

In [14]:
# ============================================================6
# 7) MANUAL LABELS
# عدليهم يدويًا بعد قراءة الـ Job Description
# 1 = relevant / 0 = not relevant
# ============================================================
semantic_labels = [1, 1, 0, 1, 0]
baseline_labels = [1, 0, 0, 1, 0]


# ============================================================
# 8) RUN ERROR ANALYSIS
# ============================================================
semantic_errors = error_analysis(
    semantic_jobs,
    semantic_labels,
    model_name="Semantic",
    top_n=len(semantic_labels)
)

baseline_errors = error_analysis(
    baseline_jobs,
    baseline_labels,
    model_name="Baseline",
    top_n=len(baseline_labels)
)

all_errors = pd.concat([semantic_errors, baseline_errors], ignore_index=True)

print("📌 Error Analysis")
display(all_errors)


# ============================================================
# 9) RUN METRICS
# ============================================================
semantic_y_pred = [1] * len(semantic_labels)
baseline_y_pred = [1] * len(baseline_labels)

semantic_metrics = classification_metrics(semantic_labels, semantic_y_pred)
baseline_metrics = classification_metrics(baseline_labels, baseline_y_pred)

metrics_df = pd.DataFrame([
    {"Model": "Semantic", **semantic_metrics},
    {"Model": "Baseline", **baseline_metrics}
])

print("📈 Metrics Comparison")
display(metrics_df)

📌 Error Analysis


,Model,Rank,Job Title,Score,Relevant?,Error Type,Reason
0,Semantic,1,Data Scientist,52.43,1,Correct,Relevant to CV based on job description
1,Semantic,2,Data Scientist,52.43,1,Correct,Relevant to CV based on job description
2,Semantic,3,Data Scientist,52.43,0,False Positive,Not relevant to CV based on job description
3,Semantic,4,Data Scientist,52.43,1,Correct,Relevant to CV based on job description
4,Semantic,5,Data Scientist,52.43,0,False Positive,Not relevant to CV based on job description
5,Baseline,1,Data Scientist,12.19,1,Correct,Relevant to CV based on job description
6,Baseline,2,Data Scientist,12.19,0,False Positive,Not relevant to CV based on job description
7,Baseline,3,Data Scientist,12.19,0,False Positive,Not relevant to CV based on job description
8,Baseline,4,Data Scientist,12.19,1,Correct,Relevant to CV based on job description
9,Baseline,5,Data Scientist,12.19,0,False Positive,Not relevant to CV based on job description


📈 Metrics Comparison


,Model,Precision,Recall,F1 Score
0,Semantic,0.6,1.0,0.7500
1,Baseline,0.4,1.0,0.5714
